# Module 4 | Class 5 Assignment: Regularization Study

**Objective:** See regularization in action by comparing unregularized, L1, and L2 logistic regression. Understand how each affects coefficients and performance.

**Dataset:** Telco Customer Churn — [Kaggle Link](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)


## Setup: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

print("All libraries imported successfully!")


## Data Preparation (Preprocessing + Split)

In [ ]:
# Load dataset
# Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Dataset loaded! Shape: {df.shape}")


In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────────

# Drop customerID
df.drop(columns=['customerID'], inplace=True)

# Fix TotalCharges (whitespace → NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Label-encode binary columns
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
               'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

# One-hot encode remaining categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f"Preprocessed shape: {df.shape}")


In [ ]:
# Define features and target
X = df.drop(columns=['Churn'])
y = df['Churn']

# Train/test split (stratified to preserve churn ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features — REQUIRED for logistic regression with regularization
scaler = StandardScaler()
X_train_sc = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)
X_test_sc = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

print(f"Train size : {X_train_sc.shape[0]}")
print(f"Test size  : {X_test_sc.shape[0]}")
print(f"Features   : {X_train_sc.shape[1]}")
print(f"Churn rate : {y_train.mean():.2%}")


## Task 1: Train Three Logistic Regression Models

We train three variants:
- **Model A** — No regularization (`penalty=None`)
- **Model B** — L1 / Lasso regularization (`penalty='l1'`, solver=`saga`)
- **Model C** — L2 / Ridge regularization (`penalty='l2'`)


In [ ]:
# Model A — No regularization
model_a = LogisticRegression(penalty=None, max_iter=1000, random_state=42)
model_a.fit(X_train_sc, y_train)
print("Model A (No Regularization) — trained!")


In [ ]:
# Model B — L1 regularization (Lasso)
model_b = LogisticRegression(penalty='l1', solver='saga', C=1.0,
                              max_iter=1000, random_state=42)
model_b.fit(X_train_sc, y_train)
print("Model B (L1 / Lasso) — trained!")


In [ ]:
# Model C — L2 regularization (Ridge)
model_c = LogisticRegression(penalty='l2', C=1.0,
                              max_iter=1000, random_state=42)
model_c.fit(X_train_sc, y_train)
print("Model C (L2 / Ridge) — trained!")


## Task 2: Compare Performance

In [ ]:
# Evaluation function
def evaluate(model, X_test, y_test, name):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'F1'       : f1_score(y_test, y_pred),
        'AUC'      : roc_auc_score(y_test, y_proba)
    }

results = [
    evaluate(model_a, X_test_sc, y_test, 'No Regularization'),
    evaluate(model_b, X_test_sc, y_test, 'L1 (Lasso)'),
    evaluate(model_c, X_test_sc, y_test, 'L2 (Ridge)'),
]

results_df = pd.DataFrame(results).set_index('Model')
print("=== Performance Comparison Table ===")
print(results_df.round(4).to_string())
results_df.round(4)


In [ ]:
# Visual comparison bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
x = np.arange(len(metrics))
width = 0.25
colors = ['#4c8be0', '#e07b54', '#5cb85c']
labels = ['No Regularization', 'L1 (Lasso)', 'L2 (Ridge)']

fig, ax = plt.subplots(figsize=(12, 6))
for i, (model_name, color) in enumerate(zip(labels, colors)):
    vals = [results_df.loc[model_name, m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=model_name,
                  color=color, edgecolor='white')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison: No Reg vs L1 vs L2', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## Task 3: Compare Coefficients

In [ ]:
# Step 1: Build coefficient comparison DataFrame
coef_df = pd.DataFrame({
    'Feature': X_train_sc.columns,
    'No Reg' : model_a.coef_[0],
    'L1'     : model_b.coef_[0],
    'L2'     : model_c.coef_[0]
}).set_index('Feature')

print("Coefficient table (first 10 features):")
print(coef_df.round(4).head(10).to_string())


In [ ]:
# Step 2: Heatmap of all coefficients
plt.figure(figsize=(10, max(8, len(coef_df) * 0.35)))
sns.heatmap(
    coef_df,
    annot=True,
    fmt='.3f',
    cmap='coolwarm',
    center=0,
    linewidths=0.4,
    annot_kws={'size': 8}
)
plt.title('Coefficient Comparison Across Regularization Types', fontsize=13, pad=15)
plt.xlabel('Model', fontsize=11)
plt.ylabel('Feature', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Step 3: Count non-zero coefficients per model
print("=== Non-zero Coefficient Counts ===")
total = len(model_a.coef_[0])
for name, model in [('No Reg', model_a), ('L1 (Lasso)', model_b), ('L2 (Ridge)', model_c)]:
    n_nonzero = np.sum(model.coef_[0] != 0)
    n_zero    = total - n_nonzero
    print(f"{name:<18}: {n_nonzero:>3} non-zero  |  {n_zero:>3} zero  (out of {total})")


In [ ]:
# Step 4: Features that L1 set to exactly zero (feature selection effect)
l1_zero_mask     = model_b.coef_[0] == 0
l1_zero_features = X_train_sc.columns[l1_zero_mask].tolist()

print(f"Features eliminated by L1 (set to zero): {len(l1_zero_features)}")
if l1_zero_features:
    for f in l1_zero_features:
        print(f"  • {f}")
else:
    print("  None — try a smaller C value (e.g. C=0.1) for stronger L1 sparsity.")


## Task 4: Vary Regularization Strength (C Values)

> **Reminder:** `C` is the *inverse* of regularization strength.
> - **Small C** → strong regularization → simpler model, smaller coefficients
> - **Large C** → weak regularization → complex model, larger coefficients


In [ ]:
# Step 1: Train L2 models across a range of C values
C_values     = [0.001, 0.01, 0.1, 1, 10, 100]
f1_scores    = []
n_large_coefs = []

for C in C_values:
    m = LogisticRegression(penalty='l2', C=C, max_iter=1000, random_state=42)
    m.fit(X_train_sc, y_train)
    y_pred = m.predict(X_test_sc)
    f1_scores.append(f1_score(y_test, y_pred))
    n_large_coefs.append(np.sum(np.abs(m.coef_[0]) > 0.1))

print("C value  |  F1 Score  |  # Large Coefs (|w|>0.1)")
print("-" * 50)
for C, f1, nc in zip(C_values, f1_scores, n_large_coefs):
    print(f"  {C:<8}|  {f1:.4f}    |  {nc}")


In [ ]:
# Step 2: Plot C vs F1 and C vs coefficient magnitude
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: C vs F1 Score
ax1.plot(C_values, f1_scores, 'o-', color='#4c8be0', lw=2, ms=8)
ax1.set_xscale('log')
ax1.set_xlabel('C  (log scale) — larger = less regularization', fontsize=11)
ax1.set_ylabel('F1 Score', fontsize=11)
ax1.set_title('C vs F1 Score (L2 Logistic Regression)', fontsize=12)
ax1.grid(linestyle='--', alpha=0.4)

# Annotate best C
best_idx = int(np.argmax(f1_scores))
ax1.annotate(
    f'Best C={C_values[best_idx]}
F1={f1_scores[best_idx]:.4f}',
    xy=(C_values[best_idx], f1_scores[best_idx]),
    xytext=(C_values[best_idx] * 3, f1_scores[best_idx] - 0.015),
    arrowprops=dict(arrowstyle='->', color='red'),
    fontsize=9, color='red'
)

# Right: C vs Number of large coefficients
ax2.plot(C_values, n_large_coefs, 'o-', color='coral', lw=2, ms=8)
ax2.set_xscale('log')
ax2.set_xlabel('C  (log scale) — larger = less regularization', fontsize=11)
ax2.set_ylabel('Number of Large Coefficients (|w| > 0.1)', fontsize=11)
ax2.set_title('C vs Coefficient Magnitude', fontsize=12)
ax2.grid(linestyle='--', alpha=0.4)

plt.suptitle('Effect of Regularization Strength (C) on L2 Logistic Regression',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Step 3: Identify the sweet-spot C value
best_C  = C_values[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Sweet-spot C value : {best_C}")
print(f"Best F1 Score      : {best_f1:.4f}")
print()
print("Interpretation:")
print(f"  C={best_C} balances model complexity and generalization.")
print("  Below this value the model is over-regularized (underfits).")
print("  Above this value large coefficients start to appear, risking overfitting.")


## Task 5: Analysis and Reflection

In [ ]:
# Final summary table
print("=" * 65)
print("             FINAL MODEL COMPARISON SUMMARY")
print("=" * 65)
print(f"{'Model':<22} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8} {'AUC':>8}")
print("-" * 65)
for _, row in results_df.iterrows():
    print(f"{row.name:<22} {row['Accuracy']:>9.4f} {row['Precision']:>10.4f} "
          f"{row['Recall']:>8.4f} {row['F1']:>8.4f} {row['AUC']:>8.4f}")
print("=" * 65)


### Written Analysis

#### What does regularization do conceptually?

Regularization is a technique that adds a **penalty term to the loss function** to prevent the model from assigning excessively large weights to any single feature. Without regularization, logistic regression can overfit the training data — the model memorizes noise instead of learning true patterns, producing very large (positive or negative) coefficients. Regularization constrains coefficient magnitudes, forcing the model to find a simpler explanation that generalizes better to unseen data.

- **L1 (Lasso)** adds a penalty proportional to the *absolute value* of each coefficient (`λ Σ|wᵢ|`). Because of its geometry, L1 naturally drives some coefficients to **exactly zero**, effectively performing automatic feature selection. This is especially useful when many features are irrelevant or redundant.
- **L2 (Ridge)** adds a penalty proportional to the *squared value* of each coefficient (`λ Σwᵢ²`). L2 shrinks all coefficients toward zero but rarely makes them exactly zero. It distributes weight more evenly across correlated features and tends to be more stable numerically.
- **C** is the inverse of the regularization strength: `C = 1/λ`. A small `C` applies strong regularization; a large `C` approaches no regularization.

#### Which model performed best on the test set?

Based on the results table above, **L2 (Ridge)** and **L1 (Lasso)** models typically outperform the unregularized model on metrics like F1 and AUC, particularly on the Churn dataset where several features are correlated (e.g., `tenure`, `MonthlyCharges`, `TotalCharges`). L2 usually edges out L1 slightly on overall accuracy, while L1 may produce a sparser and more interpretable model.

#### Which model would you deploy in production?

For a **customer churn prediction** system, I would deploy the **L2 (Ridge) Logistic Regression with C=1.0** (or the tuned sweet-spot C from Task 4) for the following reasons:

1. **Performance:** L2 achieves competitive Accuracy, F1, and AUC while avoiding overfitting.
2. **Interpretability:** Logistic regression coefficients can be directly communicated to business stakeholders — e.g., "a one-unit increase in MonthlyCharges (scaled) increases churn log-odds by X."
3. **Speed:** Both training and inference are extremely fast, suitable for real-time scoring of millions of customers.
4. **Stability:** L2 handles correlated features gracefully, unlike L1 which can arbitrarily select one feature from a correlated group and discard others.

If **feature reduction** were a priority (e.g., to reduce data collection costs), L1 would be preferred since it automatically identifies and zeroes out irrelevant features. In that scenario, L1 with a tuned `C` (e.g., C=0.1) would be the production choice.
